**1. Team Lead Extraction and Replications Calculation**

In [ ]:
import pandas as pd

# Load the replications CSV
replications = pd.read_csv("/content/WorldBank_Replications - WorldBank_Replications.csv")

# Load the main.xlsx file, replace '/path/to/your/main.xlsx' with the correct file path
main_df = pd.read_excel("/content/wbg_Updated_5.xlsx")

# Remove duplicates based on 'Project ID', keeping only the first occurrence
main_df_cleaned = main_df.drop_duplicates(subset='Project ID', keep='first')

# Step 1: Extract team lead data and calculate replication counts
team_lead_df = main_df_cleaned[['Project ID', 'Team Leads']].dropna()

# Step 2: Calculate replication pairs (team_lead_1 and team_lead_2) from 'replications'
team_lead_replications = replications[['document_id_1', 'document_id_2']]


In [ ]:

# Use .loc to safely assign values to the columns without triggering the warning
team_lead_replications.loc[:, 'team_lead_1'] = team_lead_replications['document_id_1'].map(main_df_cleaned.set_index('Project ID')['Team Leads'])
team_lead_replications.loc[:, 'team_lead_2'] = team_lead_replications['document_id_2'].map(main_df_cleaned.set_index('Project ID')['Team Leads'])

# Remove rows where 'team_lead_1' or 'team_lead_2' is NaN
team_lead_replications = team_lead_replications.dropna(subset=['team_lead_1', 'team_lead_2'])

# Step 3: Count replications per team lead pair
team_lead_counts = team_lead_replications.groupby(['team_lead_1', 'team_lead_2']).size().reset_index(name='Replication Count')

# Optional: Count replications per individual team lead
# team_lead_counts_1 = team_lead_replications['team_lead_1'].value_counts().reset_index(name='Replication Count').rename(columns={'index': 'Team Lead'})
# team_lead_counts_2 = team_lead_replications['team_lead_2'].value_counts().reset_index(name='Replication Count').rename(columns={'index': 'Team Lead'})

# Combine the counts from both team_leads (team_lead_1 and team_lead_2)
# team_lead_counts_combined = pd.concat([team_lead_counts_1, team_lead_counts_2]).groupby('Team Lead')['Replication Count'].sum().reset_index()

# Check the result
team_lead_counts.head()


<ipython-input-2-80daafcc8457>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  team_lead_replications.loc[:, 'team_lead_1'] = team_lead_replications['document_id_1'].map(main_df_cleaned.set_index('Project ID')['Team Leads'])
<ipython-input-2-80daafcc8457>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  team_lead_replications.loc[:, 'team_lead_2'] = team_lead_replications['document_id_2'].map(main_df_cleaned.set_index('Project ID')['Team Leads'])


,team_lead_1,team_lead_2,Replication Count
0,"Aanchal Anand, Kathrine M. Kelm","George Soraya, Marcus Lee, Jian Vun",1
1,"Abdalwahab Khatib, Wenye Dong","Namoos Zaheer, Kiran Afzal",1
2,Abdelghany,Anna-Maria Bogdanova; Victoria Alexeeva,1
3,Abdelghany,Artessa Saldivar-Sali,1
4,Abdelghany,"Emma Isinika Modamba, Timothy D Robertson",1


**2. Part A: Team Lead Treemap (Plotly)**

In [ ]:
import plotly.express as px
import pandas as pd

# Sort by 'Replication Count' and select top 20
top_20 = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(20)

# Clean the labels by removing everything after the comma
top_20['team_lead_1'] = top_20['team_lead_1'].str.replace(r',.*$', '', regex=True)

# Compute Replication Count Sum for each team lead
replication_sum = top_20.groupby('team_lead_1')['Replication Count'].sum().reset_index()
replication_sum.rename(columns={'Replication Count': 'Replication Count Sum'}, inplace=True)

# Merge the sum back into top_20
top_20 = pd.merge(top_20, replication_sum, on='team_lead_1', how='left')

# Treemap with color based on Replication Count Sum
fig = px.treemap(top_20,
                 path=['team_lead_1'],
                 values='Replication Count',
                 color='Replication Count Sum',
                 color_continuous_scale='Blues',
                 hover_data=['Replication Count', 'Replication Count Sum'],
                 title="Top 20 Team Lead Replication Treemap")

# Display the replication count inside each box
fig.data[0].texttemplate = "%{label}<br>%{value}"

# Update layout
fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    coloraxis_colorbar=dict(
        title="Replication<br>Count",
        tickfont=dict(size=12),
        titlefont=dict(size=14),
    )
)

fig.show()


In [ ]:
import plotly.express as px
import pandas as pd

# Sort by 'Replication Count' and select top 20
top_20 = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(22)

# Clean the labels by removing everything after the comma
top_20['team_lead_1'] = top_20['team_lead_1'].str.replace(r',.*$', '', regex=True)

# Debug print: unique team lead names in top 20
unique_team_leads = top_20['team_lead_1'].unique()
print("Unique Team Leads in Top 20:")
print(unique_team_leads)


# Compute Replication Count Sum for each team lead
replication_sum = top_20.groupby('team_lead_1')['Replication Count'].sum().reset_index()
replication_sum.rename(columns={'Replication Count': 'Replication Count Sum'}, inplace=True)

# Merge the sum back into top_20
top_20 = pd.merge(top_20, replication_sum, on='team_lead_1', how='left')

# Treemap with color based on Replication Count Sum
fig = px.treemap(top_20,
                 path=['team_lead_1'],
                 values='Replication Count',
                 color='Replication Count Sum',
                 color_continuous_scale='Blues',
                 hover_data=['Replication Count', 'Replication Count Sum'],
                 title="Top 10 Team Lead Replication Treemap")

# Display the replication count inside each box
fig.data[0].texttemplate = "%{label}<br>%{value}"

# Update layout
fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    coloraxis_colorbar=dict(
        title="Replication<br>Count",
        tickfont=dict(size=12),
        titlefont=dict(size=14),
    )
)

fig.show()


Unique Team Leads in Top 20:
['Denis Jordy' 'Natasha Beschorner' 'Joop Stoutjesdijk' 'John Virdin'
 'Madhur Gautam' 'Abdoulaye Toure' 'Masood Ahmad' 'Wael Zakout'
 'Gayatri Acharya' 'Salimata D. Follea']


**3. Part B: Top Team Lead Pairs Bar Chart (Plotly)**

In [ ]:
pair_team_leads = team_lead_counts.copy()
pair_team_leads.columns = ['team_lead_1', 'team_lead_2', 'replication_count']

# First merge: Add sector for team_lead_1
pair_team_leads = pair_team_leads.merge(
    main_df[['Team Leads', 'Sector']],
    left_on='team_lead_1',
    right_on='Team Leads',
    how='left'
).rename(columns={'Sector': 'Sector_1'}).drop(columns=['Team Leads'])

# Second merge: Add sector for team_lead_2
pair_team_leads = pair_team_leads.merge(
    main_df[['Team Leads', 'Sector']],
    left_on='team_lead_2',
    right_on='Team Leads',
    how='left'
).rename(columns={'Sector': 'Sector_2'}).drop(columns=['Team Leads'])

# Combine sectors
pair_team_leads['Sector'] = pair_team_leads['Sector_1'].fillna(pair_team_leads['Sector_2'])

# Filter top 20
top_pairs = pair_team_leads.sort_values(by='replication_count', ascending=False)

# Bar chart
import plotly.express as px

fig = px.bar(
    top_pairs,
    x='replication_count',
    y=top_pairs.apply(lambda row: f"{row['team_lead_1']} → {row['team_lead_2']}", axis=1),
    color='Sector',
    orientation='h',
    title="Top 20 Team Lead Replication Pairs by Sector",
    labels={'replication_count': 'Number of Replications'},
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(margin=dict(t=50, l=200, r=25, b=25))
fig.show()


**4. Part C: Sankey Diagram (Plotly)**

In [ ]:
import plotly.io as pio
pio.renderers.default = 'colab'  # or 'notebook' or 'iframe_connected'


In [ ]:
# Limit to top 50 most frequent team lead replications
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(50)

# Rebuild the team lead list based on top data
all_top_team_leads = pd.unique(top_team_lead_counts[['team_lead_1', 'team_lead_2']].values.ravel())
name_to_index_top = {name: idx for idx, name in enumerate(all_top_team_leads)}

# Map to Sankey indices
sources = top_team_lead_counts['team_lead_1'].map(name_to_index_top).astype(int).tolist()
targets = top_team_lead_counts['team_lead_2'].map(name_to_index_top).astype(int).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Step 1: Filter the top 10 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(15)


# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels (strip "_src"/"_tgt")
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [name.replace('_src', '').replace('_tgt', '') for name in all_ids]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map names to indices for Sankey
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(0,100,200,0.4)"
    )
)])

fig.update_layout(title_text="Top 15 Team Lead Replication Flows (Sankey Diagram)", font_size=12)
fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Step 1: Filter the top 15 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(15)

# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# 🔹 Debug print: Unique team leads involved in the Sankey diagram
unique_leads_in_sankey = pd.unique(
    pd.concat([top_team_lead_counts['team_lead_1'], top_team_lead_counts['team_lead_2']])
)
print("Unique Team Leads in Sankey Diagram:")
print(unique_leads_in_sankey)

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels (strip "_src"/"_tgt")
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [name.replace('_src', '').replace('_tgt', '') for name in all_ids]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map names to indices for Sankey
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(0,100,200,0.4)"
    )
)])

fig.update_layout(title_text="Top 15 Team Lead Replication Flows (Sankey Diagram)", font_size=12)
fig.show()


Unique Team Leads in Sankey Diagram:
['Denis Jordy' 'Natasha Beschorner' 'Joop Stoutjesdijk' 'John Virdin'
 'Madhur Gautam' 'Abdoulaye Toure' 'David Meerbach'
 'Natasha\xa0Beschorner' 'John Fraser Stewart' 'Giuseppe Fantozzi'
 'James L. Neumann' 'Harideep Singh/Ina Pranoto' 'Bérengère Prince'
 'IJsbrand H. de Jong']


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Step 1: Filter the top 15 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(22)

# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# 🔹 Normalize non-breaking spaces and strip whitespace
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace('\xa0', ' ', regex=False).str.strip()
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace('\xa0', ' ', regex=False).str.strip()

# 🔹 Debug print: Unique team leads involved in the Sankey diagram
unique_leads_in_sankey = pd.unique(
    pd.concat([top_team_lead_counts['team_lead_1'], top_team_lead_counts['team_lead_2']])
)
print("Unique Team Leads in Sankey Diagram:")
print(unique_leads_in_sankey)

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels (strip "_src"/"_tgt")
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [name.replace('_src', '').replace('_tgt', '') for name in all_ids]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map names to indices for Sankey
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(0,100,200,0.4)"
    )
)])

fig.update_layout(title_text="Top 15 Team Lead Replication Flows (Sankey Diagram)", font_size=12)
fig.show()


Unique Team Leads in Sankey Diagram:
['Denis Jordy' 'Natasha Beschorner' 'Joop Stoutjesdijk' 'John Virdin'
 'Madhur Gautam' 'Abdoulaye Toure' 'Masood Ahmad' 'Wael Zakout'
 'Gayatri Acharya' 'Salimata D. Follea' 'David Meerbach'
 'John Fraser Stewart' 'Giuseppe Fantozzi' 'James L. Neumann'
 'Harideep Singh/Ina Pranoto' 'Bérengère Prince' 'IJsbrand H. de Jong'
 'Peter Goodman' 'Mahwash Wasia' 'Enrique Pantoja'
 'Michael CarrolYMarie-HClCne Collion' 'Jean-Michel Pavy']


In [ ]:
# 🔹 Combine both directions of replication
lead_1_sums = top_team_lead_counts[['team_lead_1', 'Replication Count']].rename(columns={'team_lead_1': 'team_lead'})
lead_2_sums = top_team_lead_counts[['team_lead_2', 'Replication Count']].rename(columns={'team_lead_2': 'team_lead'})

# 🔹 Concatenate and group
all_replications = pd.concat([lead_1_sums, lead_2_sums])
replication_sums = all_replications.groupby('team_lead')['Replication Count'].sum().reset_index()

# 🔹 Sort by replication count descending
replication_sums = replication_sums.sort_values(by='Replication Count', ascending=False)

# 🔹 Print the results
print("\nReplication Sum per Team Lead:")
for _, row in replication_sums.iterrows():
    print(f"{row['team_lead']}: {row['Replication Count']}")



Replication Sum per Team Lead:
Natasha Beschorner: 54
Joop Stoutjesdijk: 47
John Virdin: 42
Denis Jordy: 30
Abdoulaye Toure: 18
David Meerbach: 11
IJsbrand H. de Jong: 10
Wael Zakout: 8
Enrique Pantoja: 8
John Fraser Stewart: 8
Madhur Gautam: 6
Bérengère Prince: 6
James L. Neumann: 6
Harideep Singh/Ina Pranoto: 6
Giuseppe Fantozzi: 6
Peter Goodman: 5
Masood Ahmad: 5
Gayatri Acharya: 4
Jean-Michel Pavy: 4
Mahwash Wasia: 4
Michael CarrolYMarie-HClCne Collion: 4
Salimata D. Follea: 4


In [ ]:
import plotly.express as px

# Step 6: Combine both source and target team leads into a single column
source_counts = top_team_lead_counts[['team_lead_1', 'Replication Count']].rename(
    columns={'team_lead_1': 'Team Lead'}
)
target_counts = top_team_lead_counts[['team_lead_2', 'Replication Count']].rename(
    columns={'team_lead_2': 'Team Lead'}
)

# Combine and group to get total involvement (as source or target)
combined_counts = pd.concat([source_counts, target_counts])
team_lead_totals = combined_counts.groupby('Team Lead', as_index=False)['Replication Count'].sum()

# Step 7: Create the Tree Map
fig_tree = px.treemap(
    team_lead_totals,
    path=['Team Lead'],
    values='Replication Count',
    color='Replication Count',
    color_continuous_scale='Blues',
    title='Total Replication Involvement per Team Lead (Tree Map)'
)

fig_tree.update_traces(root_color="lightgrey")
fig_tree.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig_tree.show()


In [ ]:
import plotly.express as px

# Step 6: Combine both source and target team leads into a single column
source_counts = top_team_lead_counts[['team_lead_1', 'Replication Count']].rename(
    columns={'team_lead_1': 'Team Lead'}
)
target_counts = top_team_lead_counts[['team_lead_2', 'Replication Count']].rename(
    columns={'team_lead_2': 'Team Lead'}
)

# Combine and group to get total involvement (as source or target)
combined_counts = pd.concat([source_counts, target_counts])
team_lead_totals = combined_counts.groupby('Team Lead', as_index=False)['Replication Count'].sum()

# Step 7: Sort by Replication Count and select top 15
team_lead_totals = team_lead_totals.sort_values(by='Replication Count', ascending=False).head(10)

# Step 8: Create the Tree Map with values displayed
fig_tree = px.treemap(
    team_lead_totals,
    path=['Team Lead'],
    values='Replication Count',
    color='Replication Count',
    color_continuous_scale='Blues',
    title='Top 10 Replication Involvement per Team Lead (Tree Map)',
    hover_data=['Replication Count']  # Display the replication count on hover
)

# Display values inside the treemap blocks
fig_tree.update_traces(textinfo="label+value")

fig_tree.update_traces(root_color="lightgrey")
fig_tree.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig_tree.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Step 1: Filter the top 15 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(10)

# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# 🔹 Normalize non-breaking spaces and strip whitespace
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace('\xa0', ' ', regex=False).str.strip()
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace('\xa0', ' ', regex=False).str.strip()

# 🔹 Combine both directions of replication for total sums
lead_1_sums = top_team_lead_counts[['team_lead_1', 'Replication Count']].rename(columns={'team_lead_1': 'team_lead'})
lead_2_sums = top_team_lead_counts[['team_lead_2', 'Replication Count']].rename(columns={'team_lead_2': 'team_lead'})

# 🔹 Concatenate and group
all_replications = pd.concat([lead_1_sums, lead_2_sums])
replication_sums = all_replications.groupby('team_lead')['Replication Count'].sum().reset_index()

# 🔹 Create a dictionary of replication sums for easy lookup
replication_dict = dict(zip(replication_sums['team_lead'], replication_sums['Replication Count']))

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels (with replication sums)
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [
    f"{name.replace('_src', '').replace('_tgt', '')} ({replication_dict.get(name.replace('_src', '').replace('_tgt', ''), 0)})"
    for name in all_ids
]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map names to indices for Sankey
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Build the Sankey diagram with hovertemplate
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(0,100,200,0.4)",
        hovertemplate=(
            "Pair: %{source} -> %{target}<br>"  # Display team leads involved
            "Replication Count: %{value}<br>"  # Replication count for the specific link
            "Total Replication Sum: %{customdata}<extra></extra>"  # Total replication sum for source node
        ),
        customdata=[replication_dict.get(name.replace('_src', '').replace('_tgt', ''), 0) for name in all_ids]  # Pass total replication sums
    )
)])

fig.update_layout(title_text="Top 10 Team Lead Replication Flows (Sankey Diagram)", font_size=12)
fig.show()




In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Step 1: Filter the top 15 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(11)

# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# 🔹 Normalize non-breaking spaces and strip whitespace
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace('\xa0', ' ', regex=False).str.strip()
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace('\xa0', ' ', regex=False).str.strip()

# 🔹 Combine both directions of replication for total sums
lead_1_sums = top_team_lead_counts[['team_lead_1', 'Replication Count']].rename(columns={'team_lead_1': 'team_lead'})
lead_2_sums = top_team_lead_counts[['team_lead_2', 'Replication Count']].rename(columns={'team_lead_2': 'team_lead'})

# 🔹 Concatenate and group by team lead to get total replication sums
all_replications = pd.concat([lead_1_sums, lead_2_sums])
replication_sums = all_replications.groupby('team_lead')['Replication Count'].sum().reset_index()

# 🔹 Create a dictionary of replication sums for easy lookup
replication_dict = dict(zip(replication_sums['team_lead'], replication_sums['Replication Count']))

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels (with replication sums)
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [
    f"{name.replace('_src', '').replace('_tgt', '')} ({replication_dict.get(name.replace('_src', '').replace('_tgt', ''), 0)})"
    for name in all_ids
]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map names to indices for Sankey
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Build the Sankey diagram with hovertemplate showing total replication sum
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(0,100,200,0.4)",
        hovertemplate=(
            "Pair: %{source} -> %{target}<br>"  # Display team leads involved
            "Replication Count: %{value}<br>"  # Replication count for the specific link
            "Total Replication Sum: %{customdata}<extra></extra>"  # Total replication sum for source node
        ),
        customdata=[replication_dict.get(name.replace('_src', '').replace('_tgt', ''), 0) for name in all_ids]  # Pass total replication sums
    )
)])

fig.update_layout(title_text="Top 15 Team Lead Replication Flows (Sankey Diagram)", font_size=12)
fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

# Step 1: Filter the top 15 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(15).copy()

# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [name.replace('_src', '').replace('_tgt', '') for name in all_ids]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map to indices
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Generate colors for nodes using Plotly's qualitative palette
unique_names = list(pd.unique(visible_labels))
color_palette = px.colors.qualitative.Plotly * (len(unique_names) // 10 + 1)
label_colors = [color_palette[unique_names.index(label)] for label in visible_labels]

# Optional: gradient link colors based on value (replication count)
import numpy as np
max_val = max(values)
min_val = min(values)
def interpolate_color(val, min_val, max_val, color_start="rgba(0,100,200,", color_end="rgba(0,200,100,", alpha=0.4):
    ratio = (val - min_val) / (max_val - min_val + 1e-6)
    r = int(0 + ratio * (0))
    g = int(100 + ratio * (100))
    b = int(200 - ratio * (100))
    return f'rgba({r},{g},{b},{alpha})'

link_colors = [interpolate_color(v, min_val, max_val) for v in values]

# Step 6: Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=20,
        thickness=18,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color=label_colors,
        hovertemplate='%{label}<extra></extra>',
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        hovertemplate='Replication Count: %{value}<extra></extra>',
    )
)])

fig.update_layout(
    title_text="Top 15 Team Lead Replication Flows",
    title_font_size=20,
    font=dict(size=13, family="Arial"),
    margin=dict(t=70, l=20, r=20, b=20),
    height=600,
)

fig.show()


<ipython-input-28-32e8a1c2eea2>:27: FutureWarning:

unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.



In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Step 1: Filter the top 15 team lead replication pairs
top_team_lead_counts = team_lead_counts.sort_values(by='Replication Count', ascending=False).head(9)

# 🔹 Clean the labels by removing everything after the comma
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace(r',.*$', '', regex=True)
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace(r',.*$', '', regex=True)

# 🔹 Normalize non-breaking spaces and strip whitespace
top_team_lead_counts['team_lead_1'] = top_team_lead_counts['team_lead_1'].str.replace('\xa0', ' ', regex=False).str.strip()
top_team_lead_counts['team_lead_2'] = top_team_lead_counts['team_lead_2'].str.replace('\xa0', ' ', regex=False).str.strip()

# 🔹 Combine both directions of replication for total sums
lead_1_sums = top_team_lead_counts[['team_lead_1', 'Replication Count']].rename(columns={'team_lead_1': 'team_lead'})
lead_2_sums = top_team_lead_counts[['team_lead_2', 'Replication Count']].rename(columns={'team_lead_2': 'team_lead'})

# 🔹 Concatenate and group
all_replications = pd.concat([lead_1_sums, lead_2_sums])
replication_sums = all_replications.groupby('team_lead')['Replication Count'].sum().reset_index()

# 🔹 Create a dictionary of replication sums for easy lookup
replication_dict = dict(zip(replication_sums['team_lead'], replication_sums['Replication Count']))

# Step 2: Create internal unique IDs for sources and targets
top_team_lead_counts['source_id'] = top_team_lead_counts['team_lead_1'] + '_src'
top_team_lead_counts['target_id'] = top_team_lead_counts['team_lead_2'] + '_tgt'

# Step 3: Build unique node ID list and their visible labels (with manual placeholder)
all_ids = pd.unique(top_team_lead_counts[['source_id', 'target_id']].values.ravel())
visible_labels = [
    f"{name.replace('_src', '').replace('_tgt', '')}    "
    for name in all_ids
]
id_to_index = {name: idx for idx, name in enumerate(all_ids)}

# Step 4: Map names to indices for Sankey
sources = top_team_lead_counts['source_id'].map(id_to_index).tolist()
targets = top_team_lead_counts['target_id'].map(id_to_index).tolist()
values = top_team_lead_counts['Replication Count'].astype(float).tolist()

# Step 5: Build the Sankey diagram with hovertemplate
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=visible_labels,
        color="lightblue"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(0,100,200,0.4)",
        hovertemplate=(
            "Pair: %{source} -> %{target}<br>"  # Display team leads involved
            "Replication Count: %{value}<br>"  # Replication count for the specific link
            "Total Replication Sum: %{customdata}<extra></extra>"  # Total replication sum for source node
        ),
        customdata=[replication_dict.get(name.replace('_src', '').replace('_tgt', ''), 0) for name in all_ids]
    )
)])

fig.update_layout(title_text="Top 10 Team Lead Replication Flows (Sankey Diagram)", font_size=12)
fig.show()
